## ITNPA4-34 Project (Q1 & Q2)

### Question 1

In [1]:
# Load Packages
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.utils import resample
import joblib

### 1.1.

In [3]:
data = pd.read_json('research_abstracts.json')
data.head()

,DOI,Abstract,Label
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical


In [4]:
# Preprocess

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    
    # Lowercase
    text = text.lower()

    # Regex
    regex_pattern = r'[^A-Za-z\s]'
    whitespace = r'\s+'

    clean_text = re.sub(regex_pattern, ' ', text)
    clean_text = re.sub(whitespace, ' ', clean_text)

    # Tokenize
    tokens = word_tokenize(clean_text)

    # Remove stop words
    clean_tokens = [token for token in tokens if token not in stop_words]

    # Lemmatize
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in clean_tokens]
    
    # Append lists
    joined_tokens = " ".join(lemmatized_tokens)

    return joined_tokens

data['clean_abstract'] = data['Abstract'].apply(preprocess)
data.head()

,DOI,Abstract,Label,clean_abstract
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai,forward looking forecasting inflation rate cou...
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai,study examined disaggregation inflation compon...
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical,paper develops dynamic factor model us euro ar...
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical,paper use simple sir macro model examine japan...
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical,backgroundgame theory useful identifying condi...


### 1.2.

Columns 'Abstract' (text features) and 'Label' (target variable) will be used from the original data. The column 'DOI' won't be used.  
Below, I use the TF-IDF feature modelling technique.

In [5]:
# Feature Extraction: TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['clean_abstract']).toarray()
y = data['Label']

X.shape

(1138, 8422)

### 1.3.

In [6]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state = 42
)

# Train baseline model (Logistic Regression)
baseline_model = LogisticRegression()
baseline_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_train = baseline_model.predict(X_train)
print(f"Training Set:\n {classification_report(y_train, y_pred_train)}")

Training Set:
               precision    recall  f1-score   support

          ai       1.00      0.47      0.64       192
   classical       0.88      1.00      0.93       718

    accuracy                           0.89       910
   macro avg       0.94      0.74      0.79       910
weighted avg       0.90      0.89      0.87       910



In [7]:
# Predict and evaluate
y_pred_test = baseline_model.predict(X_test)

# Calculate metrics
baseline_accuracy = accuracy_score(y_test, y_pred_test)
baseline_precision = precision_score(y_test, y_pred_test, average = 'macro')
baseline_recall = recall_score(y_test, y_pred_test, average = 'macro')
baseline_f1 = f1_score(y_test, y_pred_test, average = 'macro')

print(f"Test Set:\n {classification_report(y_test, y_pred_test)}")

Test Set:
               precision    recall  f1-score   support

          ai       1.00      0.37      0.54        46
   classical       0.86      1.00      0.93       182

    accuracy                           0.87       228
   macro avg       0.93      0.68      0.73       228
weighted avg       0.89      0.87      0.85       228



**Comment:**  
The baseline model achieves an overall test accuracy of 87% compared to a training accuracy of 89%. This shows stable learning and minimal overfitting on unseen data. However, the macro-averaged F1-score of 73% on the test set highlights the impact of a class imbalance (182/228 = 80% classical and 46/228 = 20% ai).  

classical: The model achieves perfect recall (1) and an F1-score of 0.93.  
ai: The model achieves perfect precision (1), indicating no false positives, but a recall of 0.37 which means 63% of true ai abstracts are misclassified as classical, resulting in a low class F1-score of 0.54.

### Question 2

### 2.1.

In [8]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Complement NB": ComplementNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Linear SVM": LinearSVC(random_state=42)
}

results = []

for name, model in models.items():
    
    #Fit model on training data
    model.fit(X_train, y_train)

    #Predictions
    y_test_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred, average = 'macro')
    recall = recall_score(y_test, y_test_pred, average = 'macro')
    f1 = f1_score(y_test, y_test_pred, average = 'macro')

    results.append({
        "Model": name,
        "Accuracy": round(accuracy, 3),
        "Precision (Macro)": round(precision, 3),
        "Recall (Macro)": round(recall, 3),
        "F1 Score (Macro)": round(f1, 3),
    })

    print(f"\n{'='*50}\nClassification Report: {name}\n{'='*50}")
    print(classification_report(y_test, y_test_pred))

# Append Baseline Model for Comparison
results.append({
        "Model": "Baseline Model",
        "Accuracy": round(baseline_accuracy, 3),
        "Precision (Macro)": round(baseline_precision, 3),
        "Recall (Macro)": round(baseline_recall, 3),
        "F1 Score (Macro)": round(baseline_f1, 3),
    })

# Display Summary Table
results_df = pd.DataFrame(results).sort_values(by="F1 Score (Macro)", ascending=False)
print("\n" + "="*50 + "\nMODEL COMPARISON SUMMARY\n" + "="*50)
print(results_df.to_string(index=False))



Classification Report: Multinomial Naive Bayes
              precision    recall  f1-score   support

          ai       1.00      0.04      0.08        46
   classical       0.81      1.00      0.89       182

    accuracy                           0.81       228
   macro avg       0.90      0.52      0.49       228
weighted avg       0.84      0.81      0.73       228


Classification Report: Complement NB
              precision    recall  f1-score   support

          ai       1.00      0.26      0.41        46
   classical       0.84      1.00      0.91       182

    accuracy                           0.85       228
   macro avg       0.92      0.63      0.66       228
weighted avg       0.87      0.85      0.81       228


Classification Report: Random Forest
              precision    recall  f1-score   support

          ai       1.00      0.59      0.74        46
   classical       0.91      1.00      0.95       182

    accuracy                           0.92       228
   m

**Comment:**  
I trained 4 machine learning classifiers on the TF-IDF feature matrix: Linear SVM, Random Forest, Complement Naive Bayes and Multinomial Naive Bayes.  
This is how they performed against the baseline Logistic Regression model.  

**Linear SVM:**  
Achieved the highest overall predictive performance with an Accuracy of 93%, a Macro Recall of 0.842 and a Macro F1-score of 0.879 (improving on the baseline Macro F1-score of 0.733 by 14.6%).  

**Random Forest:**  
Achieved the highest Precision of 0.953 and the second-highest Macro F1-score of 0.845 (improving on the baseline Macro F1-score by 11.2%).  

**Naive Bayes Models:**  
Underperformed compared to the baseline model. Multinomial Naive Bayes achieved the lowest Macro F1-score of 0.488 due to class imbalance, while Complement Naive Bayes partially corrected for it, achieving a Macro F1-score of 0.664, 6.9% lower than the baseline.  


### 2.2.

In [9]:
# Combine Training into DataFrame for Resampling
train_df = pd.DataFrame({'label': y_train.values})

# Separate majority (classical) and minority (ai) classes in the training set
df_majority = train_df[train_df['label'] == 'classical']
df_minority = train_df[train_df['label'] == 'ai']

# Extract indices
idx_majority = df_majority.index
idx_minority = df_minority.index

# Resample Minority Class (Oversampling)
idx_minority_upsampled = resample(
    idx_minority,
    replace = True,
    n_samples = len(idx_majority),  # Matches majority class count 
    random_state = 42
)

# Combine majority indices with upsampled minority indices
resampled_indices = list(idx_majority) + list(idx_minority_upsampled)

# Use resampled indices
x_train_resampled = X_train[resampled_indices]
y_train_resampled = y_train.iloc[resampled_indices]

print(f"Original Class Counts:\n{y_train.value_counts()}" )
print(f"\nResampled Class Counts:\n{y_train_resampled.value_counts()}")

# Train Baseline Model on Resampled Data
enhanced_model = LogisticRegression()
enhanced_model.fit(x_train_resampled, y_train_resampled)

# Predict
y_train_pred_enhanced = enhanced_model.predict(X_train)
y_test_pred_enhanced = enhanced_model.predict(X_test)

print("\n" + "=" * 60)
print("ENHANCED MODEL REPORT (Logistic Regression + Oversampling)")
print("=" * 60)
print(f"Training Set:\n {classification_report(y_train, y_train_pred_enhanced)}")
print(f"Test Set: \n {classification_report(y_test, y_test_pred_enhanced)}")


Original Class Counts:
Label
classical    718
ai           192
Name: count, dtype: int64

Resampled Class Counts:
Label
classical    718
ai           718
Name: count, dtype: int64

ENHANCED MODEL REPORT (Logistic Regression + Oversampling)
Training Set:
               precision    recall  f1-score   support

          ai       0.95      0.96      0.96       192
   classical       0.99      0.99      0.99       718

    accuracy                           0.98       910
   macro avg       0.97      0.97      0.97       910
weighted avg       0.98      0.98      0.98       910

Test Set: 
               precision    recall  f1-score   support

          ai       0.79      0.80      0.80        46
   classical       0.95      0.95      0.95       182

    accuracy                           0.92       228
   macro avg       0.87      0.87      0.87       228
weighted avg       0.92      0.92      0.92       228



**Comment:**  
To address the class imbalance (80% classical and 20% ai), I performed Upsampling using sklearn.utils.resample which duplicates instances from the ai training class with replacement until both classes are equally represented in the training set. 

**Comparing Test Set Performance Before and After Upsampling:**  
The test accuracy improved from 87% to 92% and the Macro F1-score improved from 0.73 to 0.87.  
For ai abstracts, the recall improved drastically from 0.37 to o.80.


### 2.3.

In [10]:
# Train best model
best_model = LinearSVC(random_state = 42)
best_model.fit(X_train, y_train)

# Save model
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib') 
joblib.dump(best_model, 'research_classifier.joblib') 
print("Model artifacts successfully saved!")

Model artifacts successfully saved!


In [11]:
%%writefile app.py
import streamlit as st
import joblib

# Load model and vectorizer
vectorizer = joblib.load('tfidf_vectorizer.joblib')
model = joblib.load('research_classifier.joblib')

# Streamlit Web UI
st.title("Researchly Research Abstract Classifier")
st.write("This app classifies research abstracts as either AI-related or classical research.")

user_abstract = st.text_area(
    "Research Abstract Input",
    height=200
)

if st.button("Classify Abstract", type="primary"):
    if user_abstract.strip() == "":
        st.warning("Please enter an abstract before submitting.")
    else:
        abstract_vector = vectorizer.transform([user_abstract])
        prediction = model.predict(abstract_vector)[0] 
        st.success(f"The abstract is classified as: {prediction}")

Writing app.py


In [56]:
!streamlit run app.py

^C
